# 第 5 章：Causal Self-Attention

这个 notebook 对应 `lessons/05_attention.md`，演示 scaled dot-product attention、causal mask、attention weights 归一化，以及禁用 mask 时未来 token 如何泄漏。

In [ ]:
import torch

from src.models.attention import (
    SingleHeadSelfAttention,
    causal_mask,
    future_attention_mass,
    scaled_dot_product_attention,
)

## 1. Causal Mask

下三角 mask 表示当前位置只能看自己和历史 token。

In [ ]:
mask = causal_mask(6)
print(mask.int())

## 2. Scaled Dot-Product Attention

`q @ k.T / sqrt(H)` 产生每个 query 对所有 key 的分数，softmax 后得到权重。

In [ ]:
torch.manual_seed(0)
q = torch.randn(1, 5, 4)
k = torch.randn(1, 5, 4)
v = torch.randn(1, 5, 4)

output = scaled_dot_product_attention(q, k, v, causal=True)
print("values:", output.values.shape)
print("weights:", output.weights.shape)
print("row sums:", output.weights.sum(dim=-1))

## 3. 未来权重为 0

在 causal attention 中，所有未来位置的 attention mass 都应为 0。

In [ ]:
print("future mass:", future_attention_mass(output.weights).item())
print(output.weights[0].round(decimals=3))

## 4. 禁用 Mask 的对照

禁用 causal mask 后，早期位置也能看到未来 token。这会让训练 loss 虚低。

In [ ]:
non_causal = scaled_dot_product_attention(q, k, v, causal=False)
print("future mass without mask:", round(future_attention_mass(non_causal.weights).item(), 4))
print(non_causal.weights[0].round(decimals=3))

## 5. 投影成 Q/K/V

真实模块会先从输入 hidden states 线性投影出 Q、K、V。

In [ ]:
attention = SingleHeadSelfAttention(input_dim=8, head_dim=4)
x = torch.randn(2, 5, 8)
projected = attention(x)
print(projected.values.shape)
print(projected.weights.shape)